# 2. Qubits, gates and entanglement

The core of the course's quantum model, in a notebook.


In [ ]:
import math
import sys
from pathlib import Path

# Make the shared qc101 library importable, whether the notebook is opened
# from the repository root or from inside the notebooks/ directory.
_here = Path.cwd().resolve()
for _candidate in (_here / "labs" / "common",
                   _here.parent / "labs" / "common",
                   _here / ".." / "labs" / "common"):
    if (_candidate / "qc101").is_dir():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise SystemExit("Could not find the qc101 library -- run this notebook "
                     "from the repository root or from notebooks/.")

from qc101 import boolean, complexnum, linalg, photon, probability, quantum, viz
from qc101.labkit import Lab

print("qc101 library ready")


## A qubit and its probabilities

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad
|\alpha|^2 + |\beta|^2 = 1$$


In [ ]:
sv = quantum.StateVector([1, 0])          # the state |0>
print(sv)
print("P(0) =", sv.probabilities()[0], " P(1) =", sv.probabilities()[1])
print("basis labels:", sv.basis_labels())

## Applying gates

A gate is a unitary matrix; applying it multiplies the state vector.


In [ ]:
sv = quantum.StateVector([1, 0])
for name, gate in (("H", quantum.H), ("X", quantum.X), ("Z", quantum.Z)):
    after = sv.apply_single(gate, 0)
    print(f"after {name}: {after}   P(0) = {after.probabilities()[0]:.4f}")

## Measuring

Measurement is the irreversible step.  It returns an outcome sampled from
the Born distribution.


In [ ]:
import random
rng = random.Random(42)
sv = quantum.StateVector([1, 0]).apply_single(quantum.H, 0)
counts = {0: 0, 1: 0}
for _ in range(2000):
    bit, collapsed = sv.measure_qubit(0, rng)
    counts[bit] += 1
print("counts:", counts)
print("the state collapsed to:", collapsed)

## Two qubits: the Bell state

H on q0 then CNOT gives $(|00\rangle + |11\rangle)/\sqrt2$.


In [ ]:
bell = quantum.bell_state()
print("state      :", bell)
print("labels     :", bell.basis_labels())
probs = bell.probabilities()
print("P(00) =", probs[0])
print("P(11) =", probs[3])

## Entanglement entropy

The entropy of the *reduced* state tells you how entangled the pair is.


In [ ]:
product = quantum.StateVector([0.5, 0.5, 0.5, 0.5])
print("Bell pair entropy    :", quantum.entanglement_entropy(bell))
print("product state entropy:", quantum.entanglement_entropy(product))
print("Bell pair is entangled:", quantum.is_entangled(bell))
print("product is entangled  :", quantum.is_entangled(product))

## The Q# ordering trap

Q# prints results right-to-left, the state vector is written left-to-right.
This is the course's 'Verifying Results' article in one cell.


In [ ]:
# q0 = 1, q1 = 0  ->  Q# says "01", the textbook says "10"
state = quantum.StateVector([0, 1, 0, 0])
print("state vector (q0 first):", state)
print("Q# would print        : '01'")